# Comparación descriptiva por tarea

Dos tareas y tres repeticiones por sistema: no se afirma superioridad general, equivalencia ni mecanismos causales. Costo incluye fallos; costo por éxito es indefinido sin éxitos o si falta consumo.

In [ ]:
from pathlib import Path
import json, os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from native_eval.bundle import COLUMNS, comparison, verify_bundle
bundle = Path(globals().get('BUNDLE', os.environ.get('NATIVE_EVAL_BUNDLE', '.runs/bundle')))
runs = pd.read_csv(bundle / 'runs.csv') if (bundle / 'runs.csv').is_file() else pd.DataFrame(columns=COLUMNS)
print('Sin resultados: no hay corridas de evaluación.' if runs.empty else f'{len(runs)} intentos observados; se muestran también los fallos.')


In [ ]:
if not runs.empty:
    display(runs)
    display(comparison(runs))
    for task, group in runs.groupby('task'):
        fig, axes = plt.subplots(1, 3, figsize=(12, 3))
        for axis, metric in zip(axes, ['input_tokens','model_steps','wall_seconds']):
            for arm, frame in group.groupby('arm'):
                axis.scatter(frame['repetition'], frame[metric], label=arm)
            axis.set(xlabel='Repetición', ylabel=metric, title=task)
            axis.legend()
        plt.tight_layout(); plt.show()
    pairs = runs.pivot(index=['task','repetition'], columns='arm', values=['success','wall_seconds'])
    if all(('success', a) in pairs.columns for a in ['agentplat','agent-teams']):
        matched = pairs.loc[pairs[('success','agentplat')].eq(True) & pairs[('success','agent-teams')].eq(True)]
        display(matched)
        print('Tiempo pareado: solo pares con éxito en ambos sistemas. La tabla completa anterior conserva timeouts y fallos.')
